# Task 1 — GET Requests Practice (JSONPlaceholder + Open-Meteo)

**Topic:** `requests` library basics — GET requests, status codes, JSON parsing, query params

**Kya seekhenge is notebook mein (what we'll learn):**
- `requests.get()` kaise use karte hain
- Response object se data kaise nikalte hain (`.status_code`, `.json()`, `.headers`)
- Query parameters kaise bhejte hain
- Path parameters (URL ke andar id) kaise use karte hain
- Basic error handling (agar request fail ho jaye)

**APIs used (dono free hain, koi API key nahi chahiye):**
- `https://jsonplaceholder.typicode.com` — fake REST API, testing ke liye best
- `https://api.open-meteo.com` — free weather API

> Har section mein pehle example diya hai (solved), phir aapke liye `# TODO` exercises hain.
> Publish karne se pehle apna naam aur GitHub link neeche README mein add kar lena.


## 0. Setup — requests install/import karna

In [1]:
# Agar requests already installed nahi hai to ye line chalayen (Jupyter mein ! se shell command chalti hai)
! pip install requests

In [2]:
import requests
# Ye check karega ke library sahi se import hui ya nahi
print("requests version:", requests.__version__)

requests version: 2.32.5


## 1. Pehla GET Request

Sabse simple cheez: server ko batao "mujhe ye data chahiye" aur wapis JSON milta hai.

Hum `jsonplaceholder.typicode.com/posts/1` se ek single post mangwa rahe hain.


In [3]:
# GET request bhej rahe hain — matlab hum server se data "mangh" rahe hain, kuch bhej nahi rahe
response = requests.get("https://jsonplaceholder.typicode.com/posts/1")

# response ek object hai — isme poora jawab (status, headers, body sab) hota hai
print(type(response))
print(response)   # ye sirf status code short form mein dikhata hai, e.g. <Response [200]>


<class 'requests.models.Response'>
<Response [200]>


## 2. Status Code aur Headers Check Karna

- `response.status_code` → number milta hai (200, 404, 500 waghera)
- `response.headers` → dictionary jaisi cheez, server ke baare mein extra info deti hai
- `response.ok` → True/False, agar status 200-399 ke beech hai to True


In [4]:
print("Status code:", response.status_code)   # 200 matlab sab theek, request kaamyab hui
print("Is it OK?  :", response.ok)             # True/False

# Content-Type batata hai server ne kis format mein data bheja hai (json, html, waghera)
print("Content-Type:", response.headers.get("Content-Type"))


Status code: 200
Is it OK?  : True
Content-Type: application/json; charset=utf-8


## 3. JSON Data Nikalna

`response.json()` HTTP response ko Python dictionary/list mein convert kar deta hai —
isse kaam karna text se bohat asaan ho jata hai.


In [6]:
data = response.json()   # ab 'data' ek Python dictionary hai, JSON text nahi

print("Poora data:", data)
print()
print("Sirf title:", data["title"])
print("Sirf body :", data["body"])
print("Sirf id :", data["id"])


Poora data: {'userId': 1, 'id': 1, 'title': 'sunt aut facere repellat provident occaecati excepturi optio reprehenderit', 'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'}

Sirf title: sunt aut facere repellat provident occaecati excepturi optio reprehenderit
Sirf body : quia et suscipit
suscipit recusandae consequuntur expedita et cum
reprehenderit molestiae ut ut quas totam
nostrum rerum est autem sunt rem eveniet architecto
Sirf id : 1


## 4. List of Items Fetch Karna aur Loop Chalana

Ab hum saare posts (100 hain total) fetch karenge aur sirf pehle 5 ke titles print karenge.


In [8]:
response = requests.get("https://jsonplaceholder.typicode.com/posts")
all_posts = response.json()   # ye ek LIST hai, dictionaries ki

print("Total posts:", len(all_posts))
print()

for post in all_posts[:5]:
    print(f"#{post['id']} -> {post['title']}")



Total posts: 100

#1 -> sunt aut facere repellat provident occaecati excepturi optio reprehenderit
#2 -> qui est esse
#3 -> ea molestias quasi exercitationem repellat qui ipsa sit aut
#4 -> eum et est occaecati
#5 -> nesciunt quas odio


## 5. Query Parameters — Data Filter Karna

Bohat saare APIs URL ke saath extra parameters accept karte hain, jese `?userId=1`.
Ye hum Python dictionary ke through bhejte hain, `requests` khud URL mein jod deta hai.


In [9]:
# Sirf userId = 1 ke posts chahiye — hum params dictionary use karenge
params = {"userId": 1}

response = requests.get("https://jsonplaceholder.typicode.com/posts", params=params)
user1_posts = response.json()

print("Actual URL jo bani:", response.url)   # dekho requests ne khud '?userId=1' add kar diya
print("User 1 ke total posts:", len(user1_posts))


Actual URL jo bani: https://jsonplaceholder.typicode.com/posts?userId=1
User 1 ke total posts: 10


## 6. Path Parameters — URL Ke Andar ID Dena

Kabhi kabhi id URL ke andar hi hoti hai, jese `/posts/1/comments` (post number 1 ke comments).


In [10]:
post_id = 1

# f-string se URL ke andar hi post_id daal rahe hain
url = f"https://jsonplaceholder.typicode.com/posts/{post_id}/comments"
response = requests.get(url)
comments = response.json()

print(f"Post {post_id} ke total comments:", len(comments))
print("Pehle comment ka email:", comments[0]["email"])


Post 1 ke total comments: 5
Pehle comment ka email: Eliseo@gardner.biz


## 7. Basic Error Handling

Har request kaamyab nahi hoti — kabhi URL galat hoti hai, kabhi server down hota hai.
Hamesha status code check karna chahiye, warna program crash ho sakta hai ya galat data mil sakta hai.


In [11]:
# Jaan bujh kar ek galat (non-existent) post id try kar rahe hain
response = requests.get("https://jsonplaceholder.typicode.com/posts/99999")

if response.status_code == 200:
    print("Mil gaya:", response.json())
else:
    # 404 ka matlab hai "Not Found" — ye post exist nahi karta
    print(f"Request fail hui! Status code: {response.status_code}")


Request fail hui! Status code: 404


## 8. Bonus — Doosri API: Weather Data (Open-Meteo)

Practice ke liye dusri API try karte hain — koi API key nahi chahiye.
Lahore ka current temperature nikalte hain.


In [14]:
# Lahore ka latitude/longitude
params = {
    "latitude": 51.5074,
    "longitude": 0.1278,
    "current_weather": True
}

response = requests.get("https://api.open-meteo.com/v1/forecast", params=params)
weather = response.json()

current = weather["current_weather"]
print(f"London ka current temperature: {current['temperature']} C")
print(f"Wind speed: {current['windspeed']} km/h")


London ka current temperature: 19.1 C
Wind speed: 13.7 km/h


---
## Exercises — Ab Aapki Bari (Your Turn)

Neeche har `# TODO` ki jagah apna code likhen. Solution comment mein hints diye hain.


### Exercise 1 — Single User Fetch Karo
User id `3` ka data `https://jsonplaceholder.typicode.com/users/3` se mangwao aur uska `name` aur `email` print karo.

In [15]:
# GET request bhej rahe hain — matlab hum server se data "mangh" rahe hain, kuch bhej nahi rahe
response = requests.get("https://jsonplaceholder.typicode.com/users/3")

# response ek object hai — isme poora jawab (status, headers, body sab) hota hai
data = response.json()   # ab 'data' ek Python dictionary hai, JSON text nahi

print("User data:")
print(f"Name: {data['name']}")
print(f"Email: {data['email']}")


User data:
Name: Clementine Bauch
Email: Nathan@yesenia.net


### Exercise 2 — Sab Users Ke Naam List Karo
`https://jsonplaceholder.typicode.com/users` se saare users fetch karo (10 hain) aur loop chala kar sirf naam print karo.

In [ ]:
# TODO: /users endpoint se GET request bhejo
# TODO: loop chalao aur har user ka sirf 'name' print karo



### Exercise 3 — Query Params Se Filter
`https://jsonplaceholder.typicode.com/comments` se sirf `postId=5` ke comments fetch karo query params use kar ke. Total count print karo.

In [ ]:
# TODO: params dictionary banao {"postId": 5}
# TODO: request bhejo aur result ki length print karo



### Exercise 4 — Apna Function Banao (Thoda Mushkil)

Ek function `get_todo(todo_id)` likho jo:
1. `https://jsonplaceholder.typicode.com/todos/{todo_id}` se GET request bheje
2. Agar status 200 ho to sirf `title` aur `completed` return kare (dictionary ki form mein)
3. Agar status 200 na ho to `None` return kare aur error print kare

Phir isko `todo_id = 7` aur `todo_id = 99999` (invalid) dono se test karo.


In [ ]:
# TODO: function get_todo(todo_id) define karo
def get_todo(todo_id):
    pass  # apna code yahan likho


# TODO: dono cases test karo


---
## Wrap-up

- `requests.get(url)` → server se data mangwana
- `response.status_code` → kaamyabi check karne ke liye
- `response.json()` → data ko Python dictionary/list mein badalna
- `params={}` → query parameters bhejne ke liye
- Path parameters → seedha URL ke andar f-string se

**Publish karne se pehle:** README mein likhen ke ye kaunsi API use hui, aur kya seekha.
